In [1]:
import torch
import pickle
from functools import lru_cache
from pathlib import Path
import numpy as np

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'


In [2]:
# Full batch mode
# for every point, there is crosspondent ray,and for every ray, it cals for every triangle in the obj,
#very time consuming
def batch_mesh_contains_points(
    ray_origins,#hand_vert
    obj_triangles,
    direction=torch.Tensor([0.4395064455, 0.617598629942, 0.652231566745]).cuda(),
):
    """Times efficient but memory greedy !
    Computes ALL ray/triangle intersections and then counts them to determine
    if point inside mesh

    Args:
    ray_origins: (batch_size x point_nb x 3)
    obj_triangles: (batch_size, triangle_nb, vertex_nb=3, vertex_coords=3)
    tol_thresh: To determine if ray and triangle are //
    Returns:
    exterior: (batch_size, point_nb) 1 if the point is outside mesh, 0 else
    """
    tol_thresh = 0.0000001
    # ray_origins.requires_grad = False
    # obj_triangles.requires_grad = False
    batch_size = obj_triangles.shape[0]
    triangle_nb = obj_triangles.shape[1]
    point_nb = ray_origins.shape[1]

    # Batch dim and triangle dim will flattened together
    batch_points_size = batch_size * triangle_nb
    # Direction is random but shared
    v0, v1, v2 = obj_triangles[:, :, 0], obj_triangles[:, :, 1], obj_triangles[:, :, 2]
    # Get edges
    v0v1 = v1 - v0
    v0v2 = v2 - v0

    # Expand needed vectors
    batch_direction = direction.view(1, 1, 3).expand(batch_size, triangle_nb, 3)

    # Compute ray/triangle intersections
    pvec = torch.cross(batch_direction, v0v2, dim=2)
    dets = torch.bmm(
        v0v1.view(batch_points_size, 1, 3), pvec.view(batch_points_size, 3, 1)
    ).view(batch_size, triangle_nb)

    # Check if ray and triangle are parallel
    parallel = abs(dets) < tol_thresh
    invdet = 1 / (dets + 0.1 * tol_thresh)

    # Repeat mesh info as many times as there are rays
    triangle_nb = v0.shape[1]
    v0 = v0.repeat(1, point_nb, 1)
    v0v1 = v0v1.repeat(1, point_nb, 1)
    v0v2 = v0v2.repeat(1, point_nb, 1)
    hand_verts_repeated = (
        ray_origins.view(batch_size, point_nb, 1, 3)
        .repeat(1, 1, triangle_nb, 1)
        .view(ray_origins.shape[0], triangle_nb * point_nb, 3)
    )
    pvec = pvec.repeat(1, point_nb, 1)
    invdet = invdet.repeat(1, point_nb)
    tvec = hand_verts_repeated - v0
    u_val = (
        torch.bmm(
            tvec.view(batch_size * tvec.shape[1], 1, 3),
            pvec.view(batch_size * tvec.shape[1], 3, 1),
        ).view(batch_size, tvec.shape[1])
        * invdet
    )
    # Check ray intersects inside triangle
    u_correct = (u_val > 0) * (u_val < 1)
    qvec = torch.cross(tvec, v0v1, dim=2)

    batch_direction = batch_direction.repeat(1, point_nb, 1)
    v_val = (
        torch.bmm(
            batch_direction.view(batch_size * qvec.shape[1], 1, 3),
            qvec.view(batch_size * qvec.shape[1], 3, 1),
        ).view(batch_size, qvec.shape[1])
        * invdet
    )
    v_correct = (v_val > 0) * (u_val + v_val < 1)
    t = (
        torch.bmm(
            v0v2.view(batch_size * qvec.shape[1], 1, 3),
            qvec.view(batch_size * qvec.shape[1], 3, 1),
        ).view(batch_size, qvec.shape[1])
        * invdet
    )
    # Check triangle is in front of ray_origin along ray direction
    t_pos = t >= tol_thresh
    parallel = parallel.repeat(1, point_nb)
    # # Check that all intersection conditions are met
    not_parallel = parallel.logical_not()
    final_inter = v_correct * u_correct * not_parallel * t_pos
    # Reshape batch point/vertices intersection matrix
    # final_intersections[batch_idx, point_idx, triangle_idx] == 1 means ray
    # intersects triangle
    final_intersections = final_inter.view(batch_size, point_nb, triangle_nb)
    # Check if intersection number accross mesh is odd to determine if point is
    # outside of mesh
    exterior = final_intersections.sum(2) % 2 == 0
    return exterior



def masked_mean_loss(dists, mask):
    mask = mask.float()
    valid_vals = mask.sum()
    if valid_vals > 0:
        loss = (mask * dists).sum() / valid_vals
    else:
        loss = torch.Tensor([0]).cuda()
    return loss


In [3]:

def batch_pairwise_dist(x, y, use_cuda=True):#  look like it calculate the distance between each point in x and each point in y
    bs, num_points_x, points_dim = x.size()
    _, num_points_y, _ = y.size()
    xx = torch.bmm(x, x.transpose(2, 1))
    yy = torch.bmm(y, y.transpose(2, 1))
    zz = torch.bmm(x, y.transpose(2, 1))
    if use_cuda:
        dtype = torch.cuda.LongTensor
    else:
        dtype = torch.LongTensor
    diag_ind_x = torch.arange(0, num_points_x).type(dtype)
    diag_ind_y = torch.arange(0, num_points_y).type(dtype)
    rx = (
        xx[:, diag_ind_x, diag_ind_x]#[barch,]
        .unsqueeze(1)
        .expand_as(zz.transpose(2, 1))
    )
    ry = yy[:, diag_ind_y, diag_ind_y].unsqueeze(1).expand_as(zz)
    P = rx.transpose(2, 1) + ry - 2 * zz
    return P


In [4]:
def batch_index_select(inp, dim, index):
    views = [inp.shape[0]] + [
        1 if i != dim else -1 for i in range(1, len(inp.shape))
    ]
    expanse = list(inp.shape)
    expanse[0] = -1
    expanse[dim] = -1
    index = index.view(views).expand(expanse)
    return torch.gather(inp, dim, index)



In [5]:


@lru_cache(maxsize=128)
def load_contacts(save_contact_paths="assets/contact_zones.pkl", display=False):
    with open(save_contact_paths, "rb") as p_f:
        contact_data = pickle.load(p_f)
    hand_verts = contact_data["verts"]
    return hand_verts, contact_data["contact_zones"]


In [6]:

def compute_contact_loss(# you have to remember, all the input is tensor with batchsize
    hand_verts_pt,
    obj_verts_pt,
    obj_faces,
    contact_thresh=0.01,# maybe the value is too big!!! you have to change it
    contact_mode="dist_sq",
    collision_thresh=10,
    collision_mode="dist_sq",
    contact_target="all",
    contact_sym=False,
    contact_zones="all",
):
    # obj_verts_pt = obj_verts_pt.detach()
    # hand_verts_pt = hand_verts_pt.detach()
    dists = batch_pairwise_dist(hand_verts_pt, obj_verts_pt) #  look like it calculate the distance between each point in hand and each point in obj euclidean distance
    #dists [b,x,y]
    mins12, min12idxs = torch.min(dists, 1)# the dim means the dim you want to reduce 
    mins21, min21idxs = torch.min(dists, 2)#the min dis between a point in hand with all points in obj

    # Get obj triangle positions
    obj_triangles = obj_verts_pt[:, obj_faces]#按照面片组合在一起[b,face_num,3,3]
    exterior = batch_mesh_contains_points(#返回的是一个mask，表示在外部的点就是1
        hand_verts_pt.detach(), obj_triangles.detach()
    )
    penetr_mask = ~exterior
    results_close = batch_index_select(obj_verts_pt, 1, min21idxs)

    if contact_target == "all":#calculate the vector between the hand and obj with the closet distances
        anchor_dists = torch.norm(results_close - hand_verts_pt, 2, 2)
    elif contact_target == "obj":
        anchor_dists = torch.norm(results_close - hand_verts_pt.detach(), 2, 2)
    elif contact_target == "hand":
        anchor_dists = torch.norm(results_close.detach() - hand_verts_pt, 2, 2)
    else:
        raise ValueError(
            "contact_target {} not in [all|obj|hand]".format(contact_target)
        )
    #choose one of the three mode to calculate the loss,the last two won't  cal the BP



    if contact_mode == "dist_sq":
        # Use squared distances to penalize contact
        if contact_target == "all":
            contact_vals = ((results_close - hand_verts_pt) ** 2).sum(2)
        elif contact_target == "obj":
            contact_vals = ((results_close - hand_verts_pt.detach()) ** 2).sum(
                2
            )
        elif contact_target == "hand":
            contact_vals = ((results_close.detach() - hand_verts_pt) ** 2).sum(
                2
            )
        else:
            raise ValueError(
                "contact_target {} not in [all|obj|hand]".format(
                    contact_target
                )
            )
        below_dist = mins21 < (contact_thresh ** 2)
    elif contact_mode == "dist":
        # Use distance to penalize contact
        contact_vals = anchor_dists
        below_dist = mins21 < contact_thresh
    elif contact_mode == "dist_tanh":
        # Use thresh * (dist / thresh) distances to penalize contact
        # (max derivative is 1 at 0)
        contact_vals = contact_thresh * torch.tanh(
            anchor_dists / contact_thresh
        )
        # All points are taken into account
        below_dist = torch.ones_like(mins21).byte()
    else:
        raise ValueError(
            "contact_mode {} not in [dist_sq|dist|dist_tanh]".format(
                contact_mode
            )
        )
    


    if collision_mode == "dist_sq":
        # Use squared distances to penalize contact
        if contact_target == "all":
            collision_vals = ((results_close - hand_verts_pt) ** 2).sum(2)
        elif contact_target == "obj":
            collision_vals = (
                (results_close - hand_verts_pt.detach()) ** 2
            ).sum(2)
        elif contact_target == "hand":
            collision_vals = (
                (results_close.detach() - hand_verts_pt) ** 2
            ).sum(2)
        else:
            raise ValueError(
                "contact_target {} not in [all|obj|hand]".format(
                    contact_target
                )
            )
    elif collision_mode == "dist":
        # Use distance to penalize collision
        collision_vals = anchor_dists
    elif collision_mode == "dist_tanh":
        # Use thresh * (dist / thresh) distances to penalize contact
        # (max derivative is 1 at 0)
        collision_vals = collision_thresh * torch.tanh(
            anchor_dists / collision_thresh
        )
    else:
        raise ValueError(
            "collision_mode {} not in "
            "[dist_sq|dist|dist_tanh]".format(collision_mode)
        )


    #begin to calculate the attr loss

    missed_mask = below_dist & exterior

    if contact_zones == "tips":
        tip_idxs = [745, 317, 444, 556, 673]#只给出了手指的五个点
        tips = torch.zeros_like(missed_mask)
        tips[:, tip_idxs] = 1
        missed_mask = missed_mask & tips
    elif contact_zones == "zones":
        _, contact_zones = load_contacts(
            "assets/contact_zones.pkl"
        )
        contact_matching = torch.zeros_like(missed_mask)
        for zone_idx, zone_idxs in contact_zones.items():
            min_zone_vals, min_zone_idxs = mins21[:, zone_idxs].min(1)
            cont_idxs = mins12.new(zone_idxs)[min_zone_idxs]
            # For each batch keep the closest point from the contact zone
            contact_matching[
                [torch.range(0, len(cont_idxs) - 1).long(), cont_idxs.long()]
            ] = 1
        missed_mask = missed_mask & contact_matching
    elif contact_zones == "all":
        missed_mask = missed_mask
    else:
        raise ValueError(
            "contact_zones {} not in [tips|zones|all]".format(contact_zones)
        )

    # Apply losses with correct mask
    missed_loss = masked_mean_loss(contact_vals, missed_mask) # attr loss,just a scalar, it has sum all the loss together
    penetr_loss = masked_mean_loss(collision_vals, penetr_mask) # penetr loss
    if contact_sym:
        obj2hand_dists = torch.sqrt(mins12)
        sym_below_dist = mins12 < contact_thresh
        sym_loss = masked_mean_loss(obj2hand_dists, sym_below_dist)
        missed_loss = missed_loss + sym_loss
    # print('penetr_nb: {}'.format(penetr_mask.sum()))
    # print('missed_nb: {}'.format(missed_mask.sum()))
    
    max_penetr_depth = (
        (anchor_dists.detach() * penetr_mask.float()).max(1)[0].mean()
    )
    mean_penetr_depth = (
        (anchor_dists.detach() * penetr_mask.float()).mean(1).mean()
    )
    contact_info = {
        "attraction_masks": missed_mask,
        "repulsion_masks": penetr_mask,
        "contact_points": results_close,
        "min_dists": mins21,
    }
    metrics = {
        "max_penetr": max_penetr_depth,
        "mean_penetr": mean_penetr_depth,
    }
    return missed_loss, penetr_loss, contact_info, metrics


In [7]:
import kornia
import torch

def transform_matrix_to_euler_angles_and_translation(matrix):
    # Convert rotation matrix to angle-axis
    quaternion = kornia.geometry.conversions.rotation_matrix_to_quaternion(matrix[:3,:3])

    # euler_angles = kornia.geometry.conversions.euler_from_quaternion(quaternion)
    euler_angles = kornia.geometry.conversions.euler_from_quaternion(quaternion[0],quaternion[1],quaternion[2],quaternion[3])

    # euler_angles = torch.Tensor(euler_angles).cuda()
    euler_angles_tensor = torch.stack(euler_angles, dim=-1)
    translation_tensor = matrix[:3,3]
    euler_and_translation = torch.stack((euler_angles_tensor,translation_tensor),dim=0).flatten()
    return euler_and_translation

def euler_angle_and_translation_to_transform_matrix(angle_and_translation):
    euler_angles = angle_and_translation[:3]
    translation = angle_and_translation[3:]
    quaternion = kornia.geometry.conversions.quaternion_from_euler(euler_angles[0],euler_angles[1],euler_angles[2])
    quaternion = torch.stack(quaternion, dim=-1)
    rotation_matrix = kornia.geometry.conversions.quaternion_to_rotation_matrix(quaternion)
    transform_matrix = torch.eye(4)
    transform_matrix[:3,:3] = rotation_matrix
    transform_matrix[:3,3] = translation
    return transform_matrix

In [8]:
import pytorch3d.io 
from pytorch3d.structures import Meshes
from pytorch3d.transforms import Rotate, Translate
import torch
from pathlib import Path

In [9]:
def optimize_object(angle_and_translation):#all     
# 设备设置 (如果有CUDA)
    target_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    folder = "/media/tony/T7/camera_data/test_object_position_optimize"
    folder = Path(folder)

    object_mesh_path = folder / Path("simplifybanana.obj")
    hand_obj_path = folder / Path("simplifyonly_hand_1015.ply")
    obj_mesh = pytorch3d.io.load_objs_as_meshes([object_mesh_path], device=target_device)


    hand_obj_IO = pytorch3d.io.IO()
    hand_obj_data = hand_obj_IO.load_mesh(path = hand_obj_path,include_textures=False,device=target_device)
    hand_vert = hand_obj_data.verts_packed().unsqueeze(0)


    #hand_vert and the obj_vert has to have the bachsize dim to use the obman func
    # 获取顶点和面。顶点需要有梯度，因为我们希望对它们进行优化。
    obj_verts = obj_mesh.verts_packed().to(target_device).unsqueeze(0)
    obj_faces = obj_mesh.faces_packed().to(target_device)

    angle_and_translation.to(target_device).requires_grad = True
    # 欧拉角和平移向量，我们将会根据它们对mesh进行变换。
    
    # 创建优化器。
    optimizer = torch.optim.Adam([angle_and_translation], lr=1e-2)

    # 迭代优化。
    for iteration in range(100):
        transform_matrix = euler_angle_and_translation_to_transform_matrix(angle_and_translation)
        optimizer.zero_grad()

        # 创建旋转和平移变换。
        R = Rotate(transform_matrix[:3,:3], device=target_device)
        T = Translate(transform_matrix[:,:3], device=target_device)

        # 应用变换。
        transformed_verts = R.transform_points(obj_verts)# you have to notice the func is done in place or not 
        transformed_verts = T.transform_points(transformed_verts)

        contact_loss, collision_loss, contact_info, metrics = compute_contact_loss(
            hand_vert, transformed_verts, obj_faces)
        # 在这里添加您的损失计算代码，例如compute_contact_loss。
        # ... (使用transformed_mesh计算loss)

        # 假设我们有一个计算出的loss。
        loss = contact_loss + collision_loss

        # 反向传播。
        loss.backward()

        # 更新欧拉角和平移向量。
        optimizer.step()

        # 打印信息或进行其他记录。
        print(f"Iteration {iteration}: Loss {loss.item()}")

    # 保存或进一步使用transformed_mesh。


In [10]:
model_position_transform = torch.tensor(
    [
    [-0.591167, -0.479865 ,-0.648268 ,1.473468],
    [0.458711, -0.861142 ,0.219134, 0.010604],
    [-0.663405, -0.167823 ,0.729198 ,0.738428],
    [0.000000 ,0.000000 ,0.000000 ,1.000000]]
    ,dtype=torch.float32)
angle_and_translation = transform_matrix_to_euler_angles_and_translation(model_position_transform)
optimize_object(angle_and_translation)

/home/tony/mine/installed/pytorch3d/pytorch3d/pytorch3d/io/obj_io.py:544: UserWarning: No mtl file provided
  warnings.warn("No mtl file provided")


RuntimeError: Expected size for first two dimensions of batch2 tensor to be: [1, 3] but got: [4, 3].